In [1]:
import numpy as np
from bokeh.plotting import figure, show
from bokeh.io import output_notebook, export_png
from bokeh.models import LinearColorMapper, ColumnDataSource, SingleIntervalTicker
from bokeh.palettes import Greys256
from bokeh.layouts import row

from sklearn.datasets import fetch_openml
import matplotlib.pyplot as plt

from selenium import webdriver as _selenium_webdriver
from selenium.webdriver.firefox.options import Options as _FirefoxOptions
from selenium.webdriver.firefox.service import Service as _FirefoxService

_opts = _FirefoxOptions()
_opts.add_argument("--headless")
_service = _FirefoxService(executable_path="/home/jet08013/anaconda3/envs/MLbook/bin/geckodriver")
_driver = _selenium_webdriver.Firefox(service=_service, options=_opts)

output_notebook()


Loading BokehJS ...

In [2]:
import numpy as np

In [3]:
x=np.linspace(0,365,366)
y=np.exp(1.3*x/366)+0.25*np.random.normal(size=366)
p=figure(title="Convolution",x_axis_label='t',y_axis_label='y')   
p.line(x,y,legend_label="Noisy Signal")
p.legend.location = "top_left"
show(p)
export_png(p, filename="/home/jet08013/GitHub/Mathematics-for-Machine-Learning/chapters/img/noisy_signal.png", webdriver=_driver)


'/home/jet08013/GitHub/Mathematics-for-Machine-Learning/chapters/img/noisy_signal.png'

In [4]:
smoothed = np.convolve(y, np.ones(10)/10, mode='valid')

In [5]:
smoothed.shape

(357,)

In [6]:
p.line(x[:len(smoothed)], smoothed, line_width=2,legend_label="Moving Average", color='red')
p.legend.location = "top_left"
show(p)
export_png(p, filename="/home/jet08013/GitHub/Mathematics-for-Machine-Learning/chapters/img/smoothed_signal.png", webdriver=_driver)


'/home/jet08013/GitHub/Mathematics-for-Machine-Learning/chapters/img/smoothed_signal.png'

In [7]:
import numpy as np
from sklearn.datasets import fetch_openml
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, LinearColorMapper
from bokeh.palettes import Greys256

# 28x28 MNIST sample
mnist = fetch_openml("mnist_784", version=1, as_frame=False)
img = mnist.data[0].reshape(28, 28).astype(np.uint8)

# Cell centers + values
xs, ys, vals, txt_color = [], [], [], []
for r in range(28):
    for c in range(28):
        v = int(img[r, c])
        xs.append(c + 0.5)
        ys.append(27 - r + 0.5)   # flip y so row 0 is top
        vals.append(v)
        txt_color.append("white" if v < 128 else "black")  # contrast text

src = ColumnDataSource(dict(x=xs, y=ys, val=vals, txt=[str(v) for v in vals], tc=txt_color))
mapper = LinearColorMapper(palette=Greys256, low=0, high=255)  # 0=black, 255=white

p = figure(width=420, height=420, x_range=(0, 28), y_range=(0, 28), toolbar_location=None)

# Color each cell by its grayscale value
p.rect(
    x="x", y="y", width=1, height=1, source=src,
    fill_color={"field": "val", "transform": mapper},
    line_color="#bdbdbd", line_width=0.5
)

# Number in each cell
p.text(
    x="x", y="y", text="txt", source=src,
    text_font_size="5pt", text_align="center", text_baseline="middle",
    text_color="tc"
)

# Clean look
p.xaxis.visible = False
p.yaxis.visible = False
p.grid.visible = False
p.outline_line_color = None

show(p)
export_png(p, filename="/home/jet08013/GitHub/Mathematics-for-Machine-Learning/chapters/img/mnist_sample.png", webdriver=_driver)


'/home/jet08013/GitHub/Mathematics-for-Machine-Learning/chapters/img/mnist_sample.png'

In [10]:
import numpy as np
from sklearn.datasets import fetch_openml
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, LinearColorMapper
from bokeh.palettes import Greys256

# Load one 28x28 MNIST image
mnist = fetch_openml("mnist_784", version=1, as_frame=False)
img = mnist.data[0].reshape(28, 28).astype(np.uint8)

# Replace each cell by floor(avg of itself + neighbors), with boundary handling
smoothed = np.zeros_like(img, dtype=np.uint8)
for r in range(28):
    for c in range(28):
        r0, r1 = max(0, r - 1), min(28, r + 2)
        c0, c1 = max(0, c - 1), min(28, c + 2)
        smoothed[r, c] = int(np.mean(img[r0:r1, c0:c1]))  # integer part

# Build Bokeh grid with colored cells + numbers
xs, ys, vals, txt_color = [], [], [], []
for r in range(28):
    for c in range(28):
        v = int(smoothed[r, c])
        xs.append(c + 0.5)
        ys.append(27 - r + 0.5)  # flip y for display
        vals.append(v)
        txt_color.append("white" if v < 128 else "black")

src = ColumnDataSource(dict(x=xs, y=ys, val=vals, txt=[str(v) for v in vals], tc=txt_color))
mapper = LinearColorMapper(palette=Greys256, low=0, high=255)

p = figure(width=420, height=420, x_range=(0, 28), y_range=(0, 28), toolbar_location=None)
p.rect(x="x", y="y", width=1, height=1, source=src,
       fill_color={"field": "val", "transform": mapper},
       line_color="#bdbdbd", line_width=0.5)
p.text(x="x", y="y", text="txt", source=src,
       text_font_size="5pt", text_align="center", text_baseline="middle",
       text_color="tc")

p.xaxis.visible = False
p.yaxis.visible = False
p.grid.visible = False
p.outline_line_color = None

show(p)
export_png(p, filename="/home/jet08013/GitHub/Mathematics-for-Machine-Learning/chapters/img/mnist_smoothed.png", webdriver=_driver)


'/home/jet08013/GitHub/Mathematics-for-Machine-Learning/chapters/img/mnist_smoothed.png'

In [ ]:
from scipy.ndimage import gaussian_filter

gaussian_smoothed = gaussian_filter(img.astype(float), sigma=(1, 1))
gaussian_smoothed = gaussian_smoothed.astype(np.uint8)

xs, ys, vals, txt_color = [], [], [], []
for r in range(28):
    for c in range(28):
        v = int(gaussian_smoothed[r, c])
        xs.append(c + 0.5)
        ys.append(27 - r + 0.5)
        vals.append(v)
        txt_color.append("white" if v < 128 else "black")

src = ColumnDataSource(dict(x=xs, y=ys, val=vals, txt=[str(v) for v in vals], tc=txt_color))
mapper = LinearColorMapper(palette=Greys256, low=0, high=255)

p = figure(width=420, height=420, x_range=(0, 28), y_range=(0, 28), toolbar_location=None)
p.rect(x="x", y="y", width=1, height=1, source=src,
       fill_color={"field": "val", "transform": mapper},
       line_color="#bdbdbd", line_width=0.5)
p.text(x="x", y="y", text="txt", source=src,
       text_font_size="5pt", text_align="center", text_baseline="middle",
       text_color="tc")

p.xaxis.visible = False
p.yaxis.visible = False
p.grid.visible = False
p.outline_line_color = None

show(p)
export_png(p, filename="/home/jet08013/GitHub/Mathematics-for-Machine-Learning/chapters/img/mnist_gaussian.png",webdriver=_driver)

RuntimeError: Neither firefox and geckodriver nor a variant of chromium browser and chromedriver are available on system PATH. You can install the former with 'conda install -c conda-forge firefox geckodriver'.

## Exponential smoothing

In [20]:
def exp_smooth(y,alpha=0.7,N=20):
    y_smooth = np.zeros_like(y)
    y_smooth[:N] = y[:N]  # start with first N values unchanged
    C = (alpha - 1)/(alpha**(N+1)-1) 
    for t in range(N, len(y)):
        for k in range(N):
            y_smooth[t] += alpha**k * y[t-k]
        y_smooth[t] *= C
    return y_smooth 

In [23]:
x=np.linspace(0,365,366)
y=np.exp(1.3*x/366)+0.25*np.random.normal(size=366)
p=figure(title="Convolution",x_axis_label='t',y_axis_label='y')   
p.line(x,y,legend_label="Noisy Signal")
p.legend.location = "top_left"
y_smooth = exp_smooth(y)
p.line(x, y_smooth, line_width=1,legend_label="Exponential Smoothing", color='red')
show(p)